# ResearchAI — Train Missing Classifier Artifacts

This notebook trains the arXiv paper category classifier.

## Instructions
1. Run all cells in order
2. Upload `paper_metadata.parquet` from `artifacts/similarity/` when prompted
3. Download `research_ai_artifacts.zip` at the end
4. Extract into your project `artifacts/` folder

## Artifacts produced
- `artifacts/classification/classifier.joblib`
- `artifacts/classification/tfidf_vectorizer.joblib`
- `artifacts/classification/labels.joblib`
- `artifacts/clustering/kmeans.joblib`
- `artifacts/clustering/cluster_assignments.parquet`
- `artifacts/clustering/cluster_terms.joblib`

In [1]:
!pip install scikit-learn joblib pandas pyarrow numpy -q
print('Dependencies installed.')

^C
Dependencies installed.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from google.colab import files
import io, pandas as pd

print('Upload: artifacts/similarity/paper_metadata.parquet')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_parquet(io.BytesIO(uploaded[fname]))
print(f'Loaded {len(df)} rows, columns: {list(df.columns)}')
print(df['broad_category'].value_counts().head(15))

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
import numpy as np

df['title'] = df['title'].fillna('')
df['abstract'] = df['abstract'].fillna('')
df['text'] = df['title'] + ' ' + df['abstract']
df = df[df['broad_category'].notna() & (df['broad_category'] != '')].copy()
print(f'Training samples: {len(df)}')

labels = sorted(df['broad_category'].unique().tolist())
print(f'Categories ({len(labels)}): {labels}')

vectorizer = HashingVectorizer(
    ngram_range=(1, 2),
    alternate_sign=False,
    lowercase=True,
    n_features=2**18,
    norm='l2',
)
print('Vectorizing...')
X = vectorizer.transform(df['text'].tolist())
y = df['broad_category'].tolist()
print(f'Feature matrix: {X.shape}')

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]}  Test: {X_test.shape[0]}')

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

clf = SGDClassifier(loss='modified_huber', max_iter=200, tol=1e-4, random_state=42, n_jobs=-1, class_weight='balanced')
print('Training SGDClassifier...')
t0 = time.time()
clf.fit(X_train, y_train)
print(f'Time: {time.time()-t0:.1f}s')
acc = accuracy_score(y_test, clf.predict(X_test))
print(f'Test accuracy: {acc:.4f}')
print(classification_report(y_test, clf.predict(X_test), zero_division=0))

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD

N_CLUSTERS = 10
print('Reducing dimensions for clustering...')
svd = TruncatedSVD(n_components=50, random_state=42)
X_r = svd.fit_transform(X)
print('Training KMeans...')
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=5)
cluster_labels = kmeans.fit_predict(X_r)
print(f'Cluster sizes: {pd.Series(cluster_labels).value_counts().sort_index().tolist()}')

In [ ]:
import os, joblib, json
from collections import Counter

os.makedirs('artifacts/classification', exist_ok=True)
os.makedirs('artifacts/clustering', exist_ok=True)

# Classification
joblib.dump(clf, 'artifacts/classification/classifier.joblib')
joblib.dump(vectorizer, 'artifacts/classification/tfidf_vectorizer.joblib')
joblib.dump(labels, 'artifacts/classification/labels.joblib')

report = {'model': 'SGDClassifier(modified_huber)', 'n_classes': len(labels), 'test_accuracy': round(acc, 4), 'classes': labels}
with open('artifacts/classification/model_report.json', 'w') as f:
    json.dump(report, f, indent=2)

# Clustering
joblib.dump(kmeans, 'artifacts/clustering/kmeans.joblib')
cluster_df = df[['id','title','broad_category']].copy()
cluster_df['cluster_id'] = cluster_labels
cluster_df.to_parquet('artifacts/clustering/cluster_assignments.parquet', index=False)

# Cluster terms
stopwords = {'the','a','an','of','in','and','to','is','for','on','with','by','from','are','was','were','be','this','that','we','our','which','as','at','it','its','has','have','been','can','not','or','but','also','based','using','proposed','paper','method','model','show','results','data','new','use','used','we','they','approach'}
cluster_terms = {}
for cid in range(N_CLUSTERS):
    mask = cluster_labels == cid
    words = ' '.join(df[mask]['text'].tolist()[:200]).lower().split()
    terms = [w for w in words if len(w)>3 and w not in stopwords]
    cluster_terms[cid] = [w for w,_ in Counter(terms).most_common(10)]
joblib.dump(cluster_terms, 'artifacts/clustering/cluster_terms.joblib')

print('All artifacts saved!')
for root, dirs, fnames in os.walk('artifacts'):
    for fn in fnames:
        fp = os.path.join(root, fn)
        print(f'  {fp}  ({os.path.getsize(fp):,} bytes)')

In [ ]:
# Smoke test
test_texts = [
    'deep learning transformer attention BERT language model NLP',
    'quantum field theory gauge symmetry particle physics',
    'stochastic differential equations probability martingale',
]
print('=== Smoke Test ===')
for text in test_texts:
    x = vectorizer.transform([text])
    pred = clf.predict(x)[0]
    proba = clf.predict_proba(x)[0]
    top3 = sorted(zip(clf.classes_, proba), key=lambda p: -p[1])[:3]
    print(f'{text[:50]}... => {pred}')
    print(f'  Top3: {[(c, round(p,3)) for c,p in top3]}')
print('Smoke test passed!')

In [ ]:
import zipfile
zip_path = '/content/research_ai_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('artifacts'):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, fp)

print(f'Archive: {zip_path}')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f'  {name}  ({info.file_size:,} bytes)')

files.download(zip_path)
print('Download started!')

## After Download

1. Extract `research_ai_artifacts.zip`
2. Copy files into your project `artifacts/` folder
3. Start Docker: `docker compose up -d`
4. Verify: visit `http://localhost:7860/health`